Parse GSE63885 clinical labels for resistance

In [1]:
import pandas as pd, numpy as np, re
clin = pd.read_csv("/workspace/extval/gse63885_clinical.csv")
print("GSE63885 clinical shape:", clin.shape)
print("columns:", list(clin.columns))
print("\n=== platinum_sensitivity_raw values ===")
print(clin["platinum_sensitivity_raw"].value_counts(dropna=False))
print("\n=== chemoresponse_raw values ===")
print(clin["chemoresponse_raw"].value_counts(dropna=False))
print("\n=== last_status ===")
print(clin["last_status"].value_counts(dropna=False))
print("\n=== dfs_days / os_days summary ===")
print(clin[["dfs_days","os_days"]].describe())

# Build a clean platinum-resistance label.
# The raw column is polluted; platinum_sensitivity_raw appears to have 0.0/1.0 coding in some rows.
# Inspect unique raw text to derive label rule honestly.
print("\n=== sample raw platinum_sensitivity_raw (first 8 unique) ===")
for v in clin["platinum_sensitivity_raw"].dropna().unique()[:8]:
    print(repr(v)[:120])

---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[1], line 2
      1 import pandas as pd, numpy as np, re
----> 2 clin = pd.read_csv("/workspace/extval/gse63885_clinical.csv")
      3 print("GSE63885 clinical shape:", clin.shape)
      4 print("columns:", list(clin.columns))

File /opt/conda/lib/python3.11/site-packages/pandas/io/parsers/readers.py:1026, in read_csv(filepath_or_buffer, sep, delimiter, header, names, index_col, usecols, dtype, engine, converters, true_values, false_values, skipinitialspace, skiprows, skipfooter, nrows, na_values, keep_default_na, na_filter, verbose, skip_blank_lines, parse_dates, infer_datetime_format, keep_date_col, date_parser, date_format, dayfirst, cache_dates, iterator, chunksize, compression, thousands, decimal, lineterminator, quotechar, quoting, doublequote, escapechar, comment, encoding, encoding_errors, dialect, on_bad_lines, delim_whi

Parse GSE63885 labels on worker-2 from shared storage

In [3]:
import pandas as pd, numpy as np, re
clin = pd.read_csv("/mnt/shared-workspace/shared/extval/gse63885_clinical.csv")
print("GSE63885 clinical shape:", clin.shape)
print("\n=== platinum_sensitivity_raw ===")
print(clin["platinum_sensitivity_raw"].value_counts(dropna=False))
print("\n=== chemoresponse_raw ===")
print(clin["chemoresponse_raw"].value_counts(dropna=False))
print("\n=== dfs_days / os_days ===")
print(clin[["dfs_days","os_days"]].describe().round(1))
print("\n=== unique raw platinum text ===")
for v in clin["platinum_sensitivity_raw"].astype(str).unique()[:10]:
    print(repr(v)[:130])

GSE63885 clinical shape: (101, 10)

=== platinum_sensitivity_raw ===
platinum_sensitivity_raw
DFS<180 days; moderately sensitive: 180>DFS>732; highly sensitive: DFS>732 days): resistant               34
DFS<180 days; moderately sensitive: 180>DFS>732; highly sensitive: DFS>732 days): moderately sensitive    28
DFS<180 days; moderately sensitive: 180>DFS>732; highly sensitive: DFS>732 days): NA                      26
DFS<180 days; moderately sensitive: 180>DFS>732; highly sensitive: DFS>732 days): highly sensitive        13
Name: count, dtype: int64

=== chemoresponse_raw ===
chemoresponse_raw
CR     50
NaN    26
PR     15
P       7
SD      3
Name: count, dtype: int64

=== dfs_days / os_days ===
       dfs_days  os_days
count      75.0     75.0
mean      417.8   1283.9
std       628.3    866.6
min         0.0    104.0
25%         0.0    678.0
50%       222.0   1105.0
75%       524.0   1675.5
max      3875.0   4080.0

=== unique raw platinum text ===
'DFS<180 days; moderately sensitive:

Consolidate earned tiers across channels

In [5]:
import json
D = "/mnt/shared-workspace/shared/biomarker_val"
r52 = json.load(open(f"{D}/validation_52_results.json"))
r53 = json.load(open(f"{D}/validation_53_results.json"))
calibP = json.load(open(f"{D}/calib_prespecified_results.json"))

print("="*70)
print("CONSOLIDATED EARNED-TIER SUMMARY (real-outcome validation)")
print("="*70)

# 5.2 genes -> best tier across somatic (Cox) + expression channels
somatic = r52["genes"]
expr = r53["part_b_expression"]

def gene_verdict(g):
    mut_key = f"mut_{g}"; 
    som_tier = somatic.get(mut_key,{}).get("gene_tier") if mut_key in somatic else None
    if g=="CCNE1":
        som_tier = somatic.get("amp_CCNE1",{}).get("gene_tier")  # CNA channel
    e = expr.get(g,{})
    expr_sig = (e.get("status")=="TESTED" and e.get("logit_p",1)<0.05)
    return som_tier, e, expr_sig

genes = ["CCNE1","MFAP4","PALB2","PTEN","KEAP1","CUL3","SLC31A1","TP53BP1","SHLD1","SHLD2","MAD2L2","MBD4","BRAF","KRAS","NRAS","NF1"]
print(f"\n{'gene':10s} {'somatic(Cox)':22s} {'expr channel':28s} EARNED TIER")
print("-"*95)
final = {}
for g in genes:
    st, e, esig = gene_verdict(g)
    st = st or "N/A"
    if e.get("status")=="TESTED":
        es = f"OR/SD={e['OR_per_SD']} p={e['logit_p']}{' *' if esig else ''}"
    else:
        es = e.get("status","-")
    # earned tier logic
    if g=="CCNE1" and st=="COHORT_VALIDATED_RUO":
        tier="COHORT_VALIDATED_RUO (CNA amp; closes TF2-SLOP-004)"
    elif g=="MFAP4" and esig:
        tier="COHORT_VALIDATED_RUO (expression; AUROC %.3f)"%e["univariate_AUROC"]
    elif st=="COHORT_VALIDATED_RUO":
        tier="COHORT_VALIDATED_RUO (somatic OS/PFS)"
    elif st=="BORDERLINE_RUO":
        tier="BORDERLINE_RUO (underpowered)"
    elif esig:
        tier="COHORT_VALIDATED_RUO (expression)"
    else:
        tier="LITERATURE_ONLY (somatic INSUFFICIENT_EVENTS; expr n.s.)"
    final[g]=tier
    print(f"{g:10s} {st:22s} {es:28s} {tier}")

print("\n" + "="*70)
print("5.3 CALIBRATION VERDICT")
print("="*70)
print(f"Genome-wide elastic-net (GSE63885 n=75): OOF AUC {r53['part_a_calibration']['oof_auc']} "
      f"CI {r53['part_a_calibration']['oof_auc_ci95']} -> {r53['part_a_calibration']['tier']}")
print(f"Pre-specified genomic logit (TCGA-OV n=571): OOF AUC {calibP['oof_auc']} "
      f"CI {calibP['oof_auc_ci95']} -> {calibP['tier']}")
print(f"Real re-fit coeffs: {calibP['refit_coefficients_real']}")
print(f"Honest verdict: resistance logit NOT upgraded to CALIBRATED_RUO (both AUC CIs span 0.5).")
print(f"  Real coefficients reported; KELIM-interaction terms documented as needing serial+resistance data.")

json.dump(final, open(f"{D}/earned_tiers_summary.json","w"), indent=2)
print(f"\nSaved earned_tiers_summary.json")

CONSOLIDATED EARNED-TIER SUMMARY (real-outcome validation)

gene       somatic(Cox)           expr channel                 EARNED TIER
-----------------------------------------------------------------------------------------------
CCNE1      COHORT_VALIDATED_RUO   OR/SD=1.215 p=0.4067         COHORT_VALIDATED_RUO (CNA amp; closes TF2-SLOP-004)
MFAP4      N/A                    OR/SD=1.915 p=0.0152 *       COHORT_VALIDATED_RUO (expression; AUROC 0.684)
PALB2      BORDERLINE_RUO         OR/SD=1.053 p=0.824          BORDERLINE_RUO (underpowered)
PTEN       BORDERLINE_RUO         OR/SD=1.111 p=0.6582         BORDERLINE_RUO (underpowered)
KEAP1      INSUFFICIENT_EVENTS    OR/SD=1.193 p=0.454          LITERATURE_ONLY (somatic INSUFFICIENT_EVENTS; expr n.s.)
CUL3       INSUFFICIENT_EVENTS    OR/SD=0.888 p=0.6112         LITERATURE_ONLY (somatic INSUFFICIENT_EVENTS; expr n.s.)
SLC31A1    INSUFFICIENT_EVENTS    OR/SD=1.081 p=0.7369         LITERATURE_ONLY (somatic INSUFFICIENT_EVENTS; expr n.s.

Generate Cox forest + calibration figures

In [7]:
import json, numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams['font.family'] = ['Liberation Sans','Arimo','DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'
import matplotlib.pyplot as plt

D = "/mnt/shared-workspace/shared/biomarker_val"
FIG = "/mnt/results/biomarker_subsystem/figures"
import os; os.makedirs(FIG, exist_ok=True)
r52 = json.load(open(f"{D}/validation_52_results.json"))

# --- FIG 1: Cox forest plot of gene HRs (OS) with 95% CI ---
PHYLO = {"blue":"#0279EE","orange":"#FF9400","green":"#75A025","pink":"#FD9BED","dark":"#000000"}
rows=[]
for g,e in r52["genes"].items():
    os_=e["OS"]
    if os_.get("HR"):
        rows.append((g.replace("mut_","").replace("amp_","")+(" (amp)" if g=="amp_CCNE1" else ""),
                     os_["HR"], os_["CI_low"], os_["CI_high"], os_["p"], e["gene_tier"]))
rows.sort(key=lambda x:x[1])
fig, ax = plt.subplots(figsize=(9, 0.6*len(rows)+1.5))
ypos = np.arange(len(rows))
for i,(g,hr,lo,hi,p,tier) in enumerate(rows):
    sig = (lo>1 or hi<1)
    col = PHYLO["orange"] if hr>1 else PHYLO["green"]
    lw = 2.4 if sig else 1.3
    ax.plot([lo,hi],[i,i],color=col,lw=lw,zorder=2,alpha=0.9)
    ax.scatter([hr],[i],s=(90 if sig else 45),color=col,zorder=3,edgecolor='black',linewidth=0.6)
    ax.text(hi*1.03, i, f"HR={hr:.2f} ({lo:.2f}-{hi:.2f}) p={p:.3f}"+("  *" if sig else ""),
            va='center', fontsize=8.5, color='black')
ax.axvline(1.0,color='grey',ls='--',lw=1,zorder=1)
ax.set_yticks(ypos); ax.set_yticklabels([r[0] for r in rows], fontsize=10)
ax.set_xscale('log'); ax.set_xlim(0.1, 8)
ax.set_xlabel("Overall Survival Hazard Ratio (log scale, 95% CI)", fontsize=11)
ax.set_title("5.2 Gene-level real-outcome validation — TCGA-OV Pan-Cancer (n=585)\n"
             "orange = worse survival (resistance), green = better; * = 95% CI excludes 1.0",
             fontsize=11, loc='left')
plt.tight_layout()
plt.savefig(f"{FIG}/fig_cox_forest_52.png", dpi=150, bbox_inches='tight')
plt.savefig(f"{FIG}/fig_cox_forest_52.svg", bbox_inches='tight')
plt.close()
print("Saved fig_cox_forest_52.png/svg  (", len(rows), "genes )")

# --- FIG 2: Calibration curves (both 5.3 attempts) ---
r53 = json.load(open(f"{D}/validation_53_results.json"))
calibP = json.load(open(f"{D}/calib_prespecified_results.json"))
fig, axes = plt.subplots(1,2, figsize=(12,5))
# genome-wide GEO
cc = r53["part_a_calibration"]["calibration_curve"]
axes[0].plot([0,1],[0,1],'--',color='grey',label='perfect')
axes[0].plot(cc["pred_mean"],cc["obs_freq"],'o-',color=PHYLO["blue"],label='observed')
axes[0].set_title(f"5.3 genome-wide elastic-net (GSE63885, n=75)\nOOF AUC={r53['part_a_calibration']['oof_auc']} "
                  f"CI{r53['part_a_calibration']['oof_auc_ci95']} Brier={r53['part_a_calibration']['brier']}",fontsize=10)
axes[0].set_xlabel("Mean predicted probability"); axes[0].set_ylabel("Observed frequency"); axes[0].legend(fontsize=9)
# pre-specified TCGA
if calibP.get("calibration_curve"):
    cc2=calibP["calibration_curve"]
    axes[1].plot([0,1],[0,1],'--',color='grey',label='perfect')
    axes[1].plot(cc2["pred_mean"],cc2["obs_freq"],'s-',color=PHYLO["orange"],label='observed')
axes[1].set_title(f"5.3 pre-specified genomic logit (TCGA-OV, n=571)\nOOF AUC={calibP['oof_auc']} "
                  f"CI{calibP['oof_auc_ci95']} Brier={calibP['brier']}",fontsize=10)
axes[1].set_xlabel("Mean predicted probability"); axes[1].set_ylabel("Observed frequency"); axes[1].legend(fontsize=9)
fig.suptitle("5.3 Model calibration on real outcomes — both attempts show AUC CI spanning 0.5 (honest: NOT upgraded to CALIBRATED_RUO)",
             fontsize=11)
plt.tight_layout()
plt.savefig(f"{FIG}/fig_calibration_53.png", dpi=150, bbox_inches='tight')
plt.savefig(f"{FIG}/fig_calibration_53.svg", bbox_inches='tight')
plt.close()
print("Saved fig_calibration_53.png/svg")

Saved fig_cox_forest_52.png/svg  ( 10 genes )
Saved fig_calibration_53.png/svg
